# C3 - Day 5: Transferability Check and Model 1b
**AnchorAI | Component 3: Cognitive Checkpointing**

Model 1 (Day 4) uses ten features from the lab experiment. Most of them (reaction times of earlier steps, the experiment number, the trial position) **do not exist in a real study/coding app**. This notebook asks two honest questions:

1. How much predictive signal remains if we keep only information a real application could know?
2. What is the best model built from the exact inputs named in **Objective 3**: *interruption duration, task complexity and prior interruption history*? We call it **Model 1b**.

Same participant folds and the same 30 test participants as before; the model type is chosen by cross-validation, not by the test score.

> Run on **CPU runtime**. Needs `samples.csv` and `meta.json` from Day 2.

## Step 0 - Mount Drive, load the Day 2 data

In [ ]:
import os, csv, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/C3'
    print('Running in Colab - Drive mounted.')
except ImportError:
    BASE = os.environ.get('C3_BASE', r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing')
    print('Running locally.')

DATA_DIR   = os.path.join(BASE, 'data', 'raw', 'hirsch')
PROC_DIR   = os.path.join(BASE, 'data', 'processed')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, ConfusionMatrixDisplay
import joblib

MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

S = pd.read_csv(os.path.join(PROC_DIR, 'samples.csv'))
with open(os.path.join(PROC_DIR, 'meta.json')) as f:
    meta = json.load(f)
LABEL_NAMES = meta['label_names']                  # 0 = Low, 1 = Medium, 2 = High

train = S[S['split'] == 'train'].reset_index(drop=True)
test  = S[S['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train):,} trials, {train.pp_id.nunique()} participants | Test: {len(test):,} trials, {test.pp_id.nunique()} participants')

## Step 1 - Which features have a counterpart in a real application?

| Lab feature | Real-application counterpart | Transferable? |
|-------------|------------------------------|---------------|
| `interruption_rt` | How long the student was away / on the interruption | **Partly** (same idea, very different scale: about 1-3 seconds in the lab, minutes or hours for students) |
| `experiment` (3 vs 4 subtasks) | Task complexity (number of steps) | **Partly** (coarse) |
| position in the session (`block`, `trial`) | How many interruptions the student already had (prior interruption history) | **Partly** |
| `sequence` (repeat vs switch) | Whether resuming means continuing the same sub-step or moving to another | Weak |
| `interruption_error`, `rt_subtask1/2`, `pre_errors`, `pp_speed` | Lab reaction times and error counts | **No** |

We build three transferable features:
* `interruption_duration` = `interruption_rt` (ms)
* `task_complexity` = number of subtasks (3 or 4)
* `prior_interruptions` = how many earlier interrupted trials the same participant had in the session (counted from the trials kept in `samples.csv`)

In [ ]:
S = S.sort_values(['pp_id', 'block', 'trial']).reset_index(drop=True)
S['interruption_duration'] = S['interruption_rt']
S['task_complexity'] = np.where(S['experiment'] == 'Exp3', 4, 3)
S['prior_interruptions'] = S.groupby('pp_id').cumcount()

train = S[S['split'] == 'train'].reset_index(drop=True)
test  = S[S['split'] == 'test'].reset_index(drop=True)
print(S[['interruption_duration', 'task_complexity', 'prior_interruptions']].describe().round(1))

## Step 2 - Feature-set comparison
We compare five feature sets with Logistic Regression and a small Random Forest (fixed settings: depth 8, at least 20 trials per leaf). For each set we report 5-fold participant-wise cross-validation and the test participants.

Reference points: random guessing gives a macro-F1 of about **0.33** for three balanced classes; the "most frequent" Dummy scores about 0.16.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

def make_model(kind, features):
    cat = [c for c in features if c == 'experiment']
    num = [c for c in features if c != 'experiment']
    parts = [('num', StandardScaler(), num)]
    if cat:
        parts.append(('cat', OneHotEncoder(handle_unknown='ignore'), cat))
    clf = {'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
           'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                                                   random_state=42, n_jobs=-1),
           'Dummy': DummyClassifier(strategy='most_frequent')}[kind]
    return Pipeline([('prep', ColumnTransformer(parts)), ('clf', clf)])

def cv_f1(kind, features):
    scores = []
    for k in sorted(train['fold'].unique()):
        tr, va = train[train['fold'] != k], train[train['fold'] == k]
        m = make_model(kind, features).fit(tr[features], tr['label'])
        scores.append(f1_score(va['label'], m.predict(va[features]), average='macro'))
    return np.mean(scores), np.std(scores, ddof=1)

def test_f1(kind, features):
    m = make_model(kind, features).fit(train[features], train['label'])
    return f1_score(test['label'], m.predict(test[features]), average='macro')

C_FEATURES = ['interruption_duration', 'task_complexity', 'prior_interruptions']
FEATURE_SETS = {
    'A duration only':                       ['interruption_duration'],
    'B + task complexity':                   ['interruption_duration', 'task_complexity'],
    'C + prior interruptions (Objective 3)': C_FEATURES,
    'D + task structure (sequence)':         C_FEATURES + ['sequence'],
    'E all 10 lab features (Model 1)':       ['sequence', 'block', 'trial', 'experiment', 'interruption_rt',
                                              'interruption_error', 'rt_subtask1', 'rt_subtask2',
                                              'pre_errors', 'pp_speed'],
}

rows = []
for name, feats in FEATURE_SETS.items():
    for kind in ['Logistic Regression', 'Random Forest']:
        m, s = cv_f1(kind, feats)
        rows.append({'feature set': name, 'model': kind, 'cv_macro_f1': m, 'cv_std': s,
                     'test_macro_f1': test_f1(kind, feats)})
dm, ds = cv_f1('Dummy', ['interruption_duration'])
results = pd.DataFrame(rows)
print(results.round(3).to_string(index=False))
print(f'\nDummy (most frequent): CV {dm:.3f}, test {test_f1("Dummy", ["interruption_duration"]):.3f} | random guessing about 0.333')

In [ ]:
best_per_set = results.sort_values('cv_macro_f1', ascending=False).groupby('feature set').head(1).sort_values('feature set')
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(best_per_set['feature set'][::-1], best_per_set['cv_macro_f1'][::-1], xerr=best_per_set['cv_std'][::-1], color='steelblue')
ax.axvline(1/3, color='black', ls=':', label='random guessing (0.33)')
ax.axvline(dm, color='gray', ls='--', label='Dummy (most frequent)')
ax.set_xlabel('Macro-F1 (5-fold participant-wise CV, best model per set)')
ax.set_title('How much signal survives when only transferable inputs are used?')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'transferability_check.png'), dpi=150)
plt.show()

## Step 3 - Model 1b: the Objective 3 inputs
Feature set **C** (interruption duration, task complexity, prior interruption history) is exactly what Objective 3 names, apart from interruption *type*, which the Hirsch data does not contain (it has a single interruption task) and which therefore cannot be learned.

We choose between Logistic Regression and Random Forest by **cross-validation**, then evaluate once on the test participants.

In [ ]:
set_c = results[results['feature set'].str.startswith('C ')].sort_values('cv_macro_f1', ascending=False)
best_kind = set_c.iloc[0]['model']
print('Chosen by CV macro-F1:', best_kind)
print(set_c[['model', 'cv_macro_f1', 'cv_std', 'test_macro_f1']].round(3).to_string(index=False))

model_1b = make_model(best_kind, C_FEATURES).fit(train[C_FEATURES], train['label'])
pred = model_1b.predict(test[C_FEATURES])
print('\nTest participants:')
print(classification_report(test['label'], pred, target_names=LABEL_NAMES, zero_division=0))
print(f'Severe errors (Low<->High): {(np.abs(test["label"].values - pred) == 2).mean():.3f}')

fig, ax = plt.subplots(figsize=(5, 4.2))
ConfusionMatrixDisplay.from_predictions(test['label'], pred, display_labels=LABEL_NAMES, cmap='Blues', ax=ax, colorbar=False)
ax.set_title(f'Model 1b ({best_kind}) - test participants')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model1b_confusion_matrix.png'), dpi=150)
plt.show()

## Step 4 - Which direction does each input push?
A simple Logistic Regression on the same three inputs shows the direction of each effect (inputs standardised). For the **High** class, a positive weight means "larger value -> a High resumption cost is more likely". This supports the rule that longer interruptions, more complex tasks and more earlier interruptions raise difficulty - or tells us honestly when they do not.

In [ ]:
lr_c = make_model('Logistic Regression', C_FEATURES).fit(train[C_FEATURES], train['label'])
weights = pd.Series(lr_c.named_steps['clf'].coef_[2], index=C_FEATURES)
print('Weights for the High class (standardised):')
print(weights.round(3).to_string())

## Step 5 - Save Model 1b and its valid input range
Lab interruptions are only seconds long. A model trained on them must **not** be fed minutes or hours directly, so the saved record includes the range of the training data. The backend has to map real durations into a safe range, or use a rule calibrated on pilot data.

In [ ]:
def predict_difficulty(model, duration_ms, task_complexity, prior_interruptions):
    X = pd.DataFrame([{'interruption_duration': duration_ms, 'task_complexity': task_complexity,
                       'prior_interruptions': prior_interruptions}])
    proba = model.predict_proba(X)[0]
    return LABEL_NAMES[int(np.argmax(proba))], dict(zip(LABEL_NAMES, proba.round(3)))

print('Example (inside the lab range):', predict_difficulty(model_1b, 2600, 4, 30))

dur = train['interruption_duration']
info = {
    'model': 'Model 1b (Objective 3 inputs)',
    'estimator': best_kind,
    'features': C_FEATURES,
    'feature_definitions': {
        'interruption_duration': 'reaction time of the interruption task in ms (lab scale)',
        'task_complexity': 'number of subtasks (3 or 4)',
        'prior_interruptions': 'earlier interrupted trials of the same participant in the session'},
    'label_names': LABEL_NAMES,
    'tercile_cut_points_ms': meta['tercile_cut_points_ms'],
    'training_ranges': {'interruption_duration_ms': [float(dur.min()), float(dur.quantile(.05)), float(dur.median()),
                                                     float(dur.quantile(.95)), float(dur.max())],
                        'task_complexity': [3, 4],
                        'prior_interruptions': [int(train['prior_interruptions'].min()), int(train['prior_interruptions'].max())]},
    'cv_macro_f1': float(set_c.iloc[0]['cv_macro_f1']),
    'test_macro_f1': float(f1_score(test['label'], pred, average='macro')),
    'caveat': 'Trained on lab data with interruptions of a few seconds; not valid for hour-long interruptions without recalibration.',
}
joblib.dump(model_1b, os.path.join(MODELS_DIR, 'c3_difficulty_model_1b.pkl'))
with open(os.path.join(OUTPUT_DIR, 'model_1b_info.json'), 'w') as f:
    json.dump(info, f, indent=2)
results.to_csv(os.path.join(OUTPUT_DIR, 'results_day5_transferability.csv'), index=False)
print(json.dumps(info, indent=2))

## Summary - Day 5 (numbers are printed above)
* **Transferability:** with only duration the models are at chance; adding task complexity, prior interruptions and (lab-specific) task structure raises the score step by step; the full lab feature set is highest.
* **Model 1b** (Objective 3 inputs) is above random guessing but modest. This is the honest strength of the transferable signal in this lab dataset.
* Interruption **type** cannot be learned from Hirsch (single type).
* Lab interruptions last seconds, so real durations need recalibration (pilot data, Objective 6) before this model drives decisions for students.

**Next: backend skeleton (checkpoint schema, API, PostgreSQL/JSONB) and a transparent difficulty rule for the prototype.**